In [1]:
import itertools
import numpy as np
from pyblock3.fcidump import FCIDUMP
from pyblock3.hamiltonian import Hamiltonian
from pyblock3.algebra.mpe import MPE
from pyblock3.algebra.symmetry import SZ
from pyscf import fci


In [2]:

L, N_UP, N_DN = 8, 4, 4     
T_HOP, U      = 1.0, 4.0
CHI           = 8     
SEED          = 0

def hubbard(L, n_up, n_dn, t_hop=1.0, u_int=4.0):
    "One-body matrix, two-body tensor and a pyblock3 Hamiltonian for an open Hubbard chain."
    h1 = np.zeros((L, L))
    i = np.arange(L - 1)
    h1[i, i + 1] = h1[i + 1, i] = -t_hop
    g2e = np.zeros((L,) * 4)
    for k in range(L):
        g2e[k, k, k, k] = u_int
    ham = Hamiltonian(FCIDUMP(pg="c1", n_sites=L, n_elec=n_up + n_dn, twos=n_up - n_dn,
                              ipg=0, h1e=h1, g2e=g2e), flat=True)
    return h1, g2e, ham


def dmrg_trial(L, n_up, n_dn, chi, sweeps=14, seed=0):
    "Return the MPS and its TRUE <Psi_T|H|Psi_T>."
    h1, g2e, ham = hubbard(L, n_up, n_dn)
    np.random.seed(seed)
    mpo, _ = ham.build_qc_mpo().compress(cutoff=1e-12)
    mps = ham.build_mps(chi)
    MPE(mps, mpo, mps).dmrg(bdims=[chi] * sweeps, noises=[1e-5] * 6 + [0],
                            dav_thrds=[1e-10], iprint=-1, n_sweeps=sweeps)

    return mps, h1, g2e, float(np.dot(mps, mpo @ mps) / np.dot(mps, mps))


mps, h1, g2e, E_T = dmrg_trial(L, N_UP, N_DN, CHI, seed=SEED)
print(f"DMRG chi={CHI}:   <Psi_T|H|Psi_T> = {E_T:.9f}")

QC MPO site   0 / 8
QC MPO site   1 / 8
QC MPO site   2 / 8
QC MPO site   3 / 8
QC MPO site   4 / 8
QC MPO site   5 / 8
QC MPO site   6 / 8
QC MPO site   7 / 8
DMRG chi=8:   <Psi_T|H|Psi_T> = -4.217221916


In [3]:
def densify(mps, L):
    "Symmetry-blocked MPS -> dense (Dl, 4, Dr) arrays, local index l = n_a + 2 n_b."
    key = lambda q: (int(q.n), int(q.twos))
    occ = lambda q: ((int(q.n) + int(q.twos)) // 2, (int(q.n) - int(q.twos)) // 2)

    def blocks(i):
        t = mps[i]
        for k in range(t.n_blocks):
            q = tuple(SZ.from_flat(int(x)) for x in t.q_labels[k])
            sh = tuple(int(x) for x in t.shapes[k])
            yield q, sh, np.asarray(t.data[t.idxs[k]:t.idxs[k + 1]]).reshape(sh)

    bonds = []                                    # bond -> {charge: (offset, size)}
    for i in range(L + 1):
        sizes = {}
        for q, sh, _ in blocks(min(i, L - 1)):
            c, d = (q[0], sh[0]) if i < L else (q[2], sh[2])
            sizes[key(c)] = d
        off, acc = {}, 0
        for c in sorted(sizes):
            off[c] = (acc, sizes[c]); acc += sizes[c]
        bonds.append((off, acc))

    out = []
    for i in range(L):
        A = np.zeros((bonds[i][1], 4, bonds[i + 1][1]))
        for (ql, qp, qr), sh, dat in blocks(i):
            na, nb = occ(qp)
            ol, dl = bonds[i][0][key(ql)]
            orr, dr = bonds[i + 1][0][key(qr)]
            A[ol:ol + dl, na + 2 * nb, orr:orr + dr] = dat[:, 0, :]
        out.append(A)
    return out


def amplitudes(ts, cfg):
    "<d|Psi_T> for a batch of configurations, by direct contraction."
    V = np.ones((cfg.shape[0], 1))
    for x, A in enumerate(ts):
        V = np.einsum("nd,ndr->nr", V, A[:, cfg[:, x], :].transpose(1, 0, 2))
    return V[:, 0]


TS = densify(mps, L)
def right_canonicalize(ts):
    "Make every site right-orthogonal. Returns the tensors and the state norm."
    ts = [t.copy() for t in ts]
    for x in range(len(ts) - 1, 0, -1):
        Dl, d, Dr = ts[x].shape
        q, r = np.linalg.qr(ts[x].reshape(Dl, d * Dr).T)
        ts[x] = q.T.reshape(q.shape[1], d, Dr)
        ts[x - 1] = np.einsum("adb,cb->adc", ts[x - 1], r)
    nrm = np.linalg.norm(ts[0])
    ts[0] /= nrm
    return ts, nrm


def perfect_sample(can, n, rng):
    "n independent configurations drawn from |<d|Psi_T>|^2, using right-canonical tensors."
    V = np.ones((n, 1))
    cfg = np.empty((n, len(can)), np.int8)
    row = np.arange(n)
    for x, A in enumerate(can):
        W = np.einsum("nd,dlr->nlr", V, A)          # (n, 4, Dr)
        p = (W ** 2).sum(-1)                        # conditionals; already sum to 1
        c = np.cumsum(p, axis=1)
        l = (rng.random((n, 1)) * c[:, -1:] < c).argmax(axis=1)
        cfg[:, x] = l
        V = W[row, l] / np.sqrt(p[row, l])[:, None]
    return cfg



def sample_msd(can, ts, n_samples, rng):
    "Distinct determinants and their exact coefficients, in all-alpha-then-all-beta order."
    cfg = perfect_sample(can, n_samples, rng)
    uniq, hits = np.unique(cfg, axis=0, return_counts=True)
    occ_a = [np.flatnonzero(u % 2 == 1) for u in uniq]
    occ_b = [np.flatnonzero(u // 2 == 1) for u in uniq]
    sign = np.array([(-1.0) ** sum(int((b < i).sum()) for i in a)
                     for a, b in zip(occ_a, occ_b)])
    return np.array(occ_a), np.array(occ_b), sign * amplitudes(ts, uniq), hits

h2e = fci.direct_spin1.absorb_h1e(h1, g2e, L, (N_UP, N_DN), 0.5)
apply_H = lambda c: fci.direct_spin1.contract_2e(h2e, c, L, (N_UP, N_DN))

_stra = fci.cistring.make_strings(range(L), N_UP)
_strb = fci.cistring.make_strings(range(L), N_DN)
_addr_a = {int(s): i for i, s in enumerate(_stra)}
_addr_b = {int(s): i for i, s in enumerate(_strb)}
_bits = lambda occ, tab: np.array([tab[int(sum(1 << int(o) for o in r))] for r in occ])
_LOW = np.tril(np.ones((L, L), np.int64), -1)       # LOW[i, j] = 1 iff j < i


def coeffs_at(ia, ib):
    "Exact MPS coefficients c_d for the determinants at civec positions (ia, ib)."
    sa, sb = np.asarray(_stra)[ia], np.asarray(_strb)[ib]
    x = np.arange(L)
    oa = ((sa[:, None] >> x) & 1).astype(np.int64)   # binary occupations
    ob = ((sb[:, None] >> x) & 1).astype(np.int64)
    cfg = (oa + 2 * ob).astype(np.int8)
    sgn = 1 - 2 * (np.einsum("nx,xy,ny->n", oa, _LOW, ob) & 1)   # interleaving sign
    return sgn * amplitudes(TS, cfg)


def energy_msd(occ_a, occ_b, coeff):
    "Energy of the sampled MSD."
    T = np.zeros((len(_stra), len(_strb)))
    T[_bits(occ_a, _addr_a), _bits(occ_b, _addr_b)] = coeff
    nrm = float((T ** 2).sum())

    HT = apply_H(T)                                  # pyscf applies H to |Psi~>
    e_var = float((T * HT).sum() / nrm)              # <Psi~|H|Psi~> / <Psi~|Psi~>
    return e_var,nrm

In [4]:
CAN, NRM = right_canonicalize(TS)

print(f"{'N draws':>9} {'n_dets':>7} {'weight':>8} | {'VARIATIONAL':>13}")
for N in (2_000, 20_000, 200_000, 1_000_000):
    oa, ob, coeff, hits = sample_msd(CAN, TS, N, np.random.default_rng(1))
    e_var, nrm = energy_msd(oa, ob, coeff)
    print(f"{N:>9} {len(coeff):>7} {nrm:>8.5f} | {e_var:>13.6f}")
print(f"\nexact <Psi_T|H|Psi_T> = {E_T:.6f}")

  N draws  n_dets   weight |   VARIATIONAL
     2000     490  0.87438 |     -3.274073
    20000    1217  0.97897 |     -4.015949
   200000    2087  0.99845 |     -4.199014
  1000000    2501  0.99980 |     -4.214759

exact <Psi_T|H|Psi_T> = -4.217222


In [5]:
def pack(cfg):
    "Exact key per configuration, two bits per site. (cfg @ 4**arange(L) overflows at L>=33.)"
    m, n = cfg.shape
    a = np.zeros((m, n + (-n) % 4), np.uint8)
    a[:, :n] = cfg
    g = a.reshape(m, -1, 4).astype(np.uint16)
    return (g * np.array([1, 4, 16, 64], np.uint16)).sum(2).astype(np.uint8)


def apply_H_sparse(occ_a, occ_b, coeff, h1, u_int, L):
    "H|Psi~> as (configs, values). Never allocates a full CI vector."
    n = len(coeff)
    cfg = np.zeros((n, L), np.int8)
    r = np.arange(n)[:, None]
    cfg[r, occ_a] += 1
    cfg[r, occ_b] += 2

    tgt = [cfg]                                          # interaction: diagonal
    val = [u_int * (cfg == 3).sum(1) * coeff]
    for p, q in np.argwhere(h1 != 0):                    # move one electron q -> p
        lo, hi = min(p, q), max(p, q)
        for bit in (1, 2):                               # alpha, then beta
            occ = (cfg & bit) > 0
            sel = np.flatnonzero(occ[:, q] & ~occ[:, p])
            if not len(sel):
                continue
            sub = cfg[sel].copy()
            sub[:, q] -= bit
            sub[:, p] += bit
            between = occ[sel][:, lo + 1:hi].sum(1)      # general Jordan-Wigner sign
            tgt.append(sub)
            val.append(h1[p, q] * (1 - 2 * (between & 1)) * coeff[sel])

    C, V = np.concatenate(tgt), np.concatenate(val)
    keys = pack(C)
    uk, first, inv = np.unique(keys, axis=0, return_index=True, return_inverse=True)
    out = np.zeros(len(uk))
    np.add.at(out, inv, V)                               # merge duplicate targets
    return C[first], out


def energies_sparse(occ_a, occ_b, coeff, ts, h1, u_int, L):
    "Variational and mixed energies, with no full CI vector anywhere."
    n = len(coeff)
    cfg = np.zeros((n, L), np.int8)
    r = np.arange(n)[:, None]
    cfg[r, occ_a] += 1
    cfg[r, occ_b] += 2
    nrm = float(coeff @ coeff)

    tc, tv = apply_H_sparse(occ_a, occ_b, coeff, h1, u_int, L)

    # variational: keep only targets that are themselves in S
    ks, kt = pack(cfg), pack(tc)
    kv_s = np.ascontiguousarray(ks).view([("", np.uint8)] * ks.shape[1]).ravel()
    kv_t = np.ascontiguousarray(kt).view([("", np.uint8)] * kt.shape[1]).ravel()
    order = np.argsort(kv_s, kind="stable")
    pos = np.clip(np.searchsorted(kv_s[order], kv_t), 0, len(order) - 1)
    hit = kv_s[order][pos] == kv_t
    e_var = float(tv[hit] @ coeff[order[pos[hit]]] / nrm)

    # mixed: every target, with its exact MPS coefficient
    oa = (tc & 1).astype(np.int64)
    ob = (tc >> 1).astype(np.int64)
    LOW = np.tril(np.ones((L, L), np.int64), -1)
    sgn = 1 - 2 * (np.einsum("nx,xy,ny->n", oa, LOW, ob) & 1)
    e_mix = float(tv @ (sgn * amplitudes(ts, tc)) / nrm)
    return e_var, e_mix, len(tc)


print(f"{'N':>8} | {'Energy':>26} ")
print(f"{'':>8} | {'pyscf':>12} {'sparse':>13} ")
for N in (2_000, 20_000, 200_000):
    oa, ob, c, _ = sample_msd(CAN, TS, N, np.random.default_rng(1))
    pv,_ = energy_msd(oa, ob, c)
    sv,_,_  = energies_sparse(oa, ob, c, TS, h1, U, L)
    print(f"{N:>8} | {pv:>12.8f} {sv:>13.8f}")

       N |                     Energy 
         |        pyscf        sparse 
    2000 |  -3.27407317   -3.27407317
   20000 |  -4.01594864   -4.01594864
  200000 |  -4.19901436   -4.19901436


In [6]:
L2, CHI2 = 20, 16
mps2, h1_2, g2e_2, E_T2 = dmrg_trial(L2, L2 // 2, L2 // 2, CHI2, seed=SEED)
TS2 = densify(mps2, L2)
CAN2, _ = right_canonicalize(TS2)
print(f"L={L2}  chi={CHI2}   <Psi_T|H|Psi_T> = {E_T2:.9f}\n")

print(f"{'N draws':>9} {'n_dets':>7} {'weight':>9} | {'VARIATIONAL':>13} {'MIXED':>13}")
for N in (5_000, 10_000, 20_000,50_000,200_000):
    oa, ob, c, _ = sample_msd(CAN2, TS2, N, np.random.default_rng(1))
    ev, em, nt = energies_sparse(oa, ob, c, TS2, h1_2, U, L2)
    print(f"{N:>9} {len(c):>7} {float(c @ c):>9.5f} | {ev:>13.6f} {em:>13.6f}")
print(f"\nexact <Psi_T|H|Psi_T> = {E_T2:.6f}     E_HF = -5.851942")

QC MPO site   0 / 20
QC MPO site   1 / 20
QC MPO site   2 / 20
QC MPO site   3 / 20
QC MPO site   4 / 20
QC MPO site   5 / 20
QC MPO site   6 / 20
QC MPO site   7 / 20
QC MPO site   8 / 20
QC MPO site   9 / 20
QC MPO site  10 / 20
QC MPO site  11 / 20
QC MPO site  12 / 20
QC MPO site  13 / 20
QC MPO site  14 / 20
QC MPO site  15 / 20
QC MPO site  16 / 20
QC MPO site  17 / 20
QC MPO site  18 / 20
QC MPO site  19 / 20
L=20  chi=16   <Psi_T|H|Psi_T> = -11.089920360

  N draws  n_dets    weight |   VARIATIONAL         MIXED
     5000    4883   0.04341 |     -0.298628    -11.109995
    10000    9585   0.07189 |     -0.972717    -11.108987
    20000   18673   0.10654 |     -1.613932    -11.108200
    50000   44469   0.16975 |     -2.577954    -11.106361
   200000  157485   0.29825 |     -4.173909    -11.104499

exact <Psi_T|H|Psi_T> = -11.089920     E_HF = -5.851942


In [7]:
from pyscf.fci import selected_ci as sci

_strbits = lambda occ: np.array([int(sum(1 << int(o) for o in r)) for r in occ], dtype=np.int64)


def sci_size(occ_a, occ_b):
    "Shape and footprint of the selected-CI array these determinants would need."
    na_, nb_ = len(np.unique(_strbits(occ_a))), len(np.unique(_strbits(occ_b)))
    return na_, nb_, na_ * nb_ * 8 / 1e9


def energy_selected_ci(occ_a, occ_b, coeff, h1_mat, g2e_ten, L, nelec):
    "<Psi~|H|Psi~> / <Psi~|Psi~>, with pyscf applying H to a selected-CI vector."
    h2e = fci.direct_spin1.absorb_h1e(h1_mat, g2e_ten, L, nelec, 0.5)
    ua, ia = np.unique(_strbits(occ_a), return_inverse=True)   # np.unique returns them SORTED,
    ub, ib = np.unique(_strbits(occ_b), return_inverse=True)   # which is what selected_ci wants
    ci = np.zeros((len(ua), len(ub)))
    ci[ia, ib] = coeff
    hc = sci.contract_2e(h2e, sci._as_SCIvector(ci, (ua, ub)), L, nelec)
    return float((np.asarray(ci) * np.asarray(hc)).sum() / (ci ** 2).sum())

In [8]:
import time

print(f"{'N':>7} {'n_dets':>7} {'weight':>9} | {'selected_ci':>14} {'sparse':>14} {'t[s]':>7}")
for N in (1_000, 5_000, 10_000):         
    oa, ob, c, _ = sample_msd(CAN2, TS2, N, np.random.default_rng(1))
    # t0 = time.perf_counter()
    # e_sci = energy_selected_ci(oa, ob, c, h1_2, g2e_2, L2, (L2 // 2, L2 // 2))
    # dt = time.perf_counter() - t0
#     # e_spa, _, _ = energies_sparse(oa, ob, c, TS2, h1_2, U, L2)
#     print(f"{N:>7} {len(c):>7} {float(c @ c):>9.5f} | {e_sci:>14.9f} {e_spa:>14.9f} {dt:>7.1f}")
# print(f"\nexact <Psi_T|H|Psi_T> = {E_T2:.9f}")

      N  n_dets    weight |    selected_ci         sparse    t[s]
